In [1]:
#Task 1
import numpy as np

def quantize_int8(data):

    min_val = min(data)
    max_val = max(data)

    scale = (max_val - min_val) / 255

    zero_point = round(-128 - (min_val / scale))

    quantized = []

    for x in data:
        q = round(x / scale + zero_point)
        q = max(-128, min(127, q))
        quantized.append(q)

    return quantized, scale, zero_point


# Test the function
data = [-1.0, -0.5, 0.0, 0.5, 1.0]

result, scale, zero_point = quantize_int8(data)

print("Original values:", data)
print("Quantized values:", result)
print("Scale:", scale)
print("Zero point:", zero_point)

Original values: [-1.0, -0.5, 0.0, 0.5, 1.0]
Quantized values: [-128, -64, 0, 64, 127]
Scale: 0.00784313725490196
Zero point: 0


The floating-point values were successfully converted into INT8 values. The quantized values have lower precision than the original values, but require less memory for storage.

In [3]:
#Task 2 -
def model_size(params, dtype):

    if dtype == "fp32":
        bytes = 4
    elif dtype == "fp16":
        bytes = 2
    elif dtype == "int8":
        bytes = 1

    total = params * bytes
    mb = total / (1024 * 1024)
    gb = total / (1024 * 1024 * 1024)

    return mb, gb


models = [125000000, 1000000000, 7000000000]

for model in models:
    print("\nParameters:", model)

    for dtype in ["fp32", "fp16", "int8"]:
        mb, gb = model_size(model, dtype)
        print(dtype, "=", round(mb, 2), "MB |", round(gb, 2), "GB")


Parameters: 125000000
fp32 = 476.84 MB | 0.47 GB
fp16 = 238.42 MB | 0.23 GB
int8 = 119.21 MB | 0.12 GB

Parameters: 1000000000
fp32 = 3814.7 MB | 3.73 GB
fp16 = 1907.35 MB | 1.86 GB
int8 = 953.67 MB | 0.93 GB

Parameters: 7000000000
fp32 = 26702.88 MB | 26.08 GB
fp16 = 13351.44 MB | 13.04 GB
int8 = 6675.72 MB | 6.52 GB


FP32 requires more memory to store model parameters. FP16 reduces the memory by half, while INT8 reduces it by almost 75%. Therefore, lower precision formats help reduce model storage requirements.

**Task 3** –
1. Where should the model be split?

First, split the model across the 8 GPUs within the same node. If the model still doesn't fit, distribute the remaining parts across multiple nodes.

2. Role of NVLink

NVLink provides high-speed communication between GPUs within the same node. It helps GPUs exchange data quickly.

3. Role of InfiniBand

InfiniBand provides high-speed communication between different nodes. It allows GPUs on separate nodes to exchange model data.

4. What happens if the roles are swapped?

If communication between GPUs in the same node uses InfiniBand instead of NVLink, performance may decrease. NVLink is designed for fast GPU-to-GPU communication within a node, while InfiniBand connects separate nodes.

**Task 4 -**

In [5]:
def check_inference(flops, compute, memory_bytes, bandwidth):

    compute_time = flops / compute
    memory_time = memory_bytes / bandwidth

    if compute_time > memory_time:
        return "Compute-bound"
    else:
        return "Memory-bound"


print(check_inference(100e9, 100e12, 20e9, 1e12))
print(check_inference(500e9, 100e12, 2e9, 1e12))
print(check_inference(200e9, 100e12, 5e9, 1e12))

Memory-bound
Compute-bound
Memory-bound
